In [2]:
from pathlib import Path

data_dir = Path("../Data")

parquet_files = list(data_dir.glob("*.parquet"))

print("Dataset directory:", data_dir.resolve())
print("Total Parquet files:", len(parquet_files))

for file in sorted(parquet_files):
    print(file.name, "-", round(file.stat().st_size / (1024**2), 2), "MB")

print("Total dataset size:", round(
    sum(f.stat().st_size for f in parquet_files) / (1024**3), 2
), "GB")

Dataset directory: C:\Users\Umesh eshwar reddy\Desktop\p4\Data
Total Parquet files: 17
test-00000-of-00001.parquet - 81.37 MB
train-00000-of-00016.parquet - 352.17 MB
train-00001-of-00016.parquet - 451.35 MB
train-00002-of-00016.parquet - 441.0 MB
train-00003-of-00016.parquet - 378.59 MB
train-00004-of-00016.parquet - 381.89 MB
train-00005-of-00016.parquet - 394.43 MB
train-00006-of-00016.parquet - 479.4 MB
train-00007-of-00016.parquet - 376.69 MB
train-00008-of-00016.parquet - 348.53 MB
train-00009-of-00016.parquet - 362.03 MB
train-00010-of-00016.parquet - 383.66 MB
train-00011-of-00016.parquet - 375.72 MB
train-00012-of-00016.parquet - 400.45 MB
train-00013-of-00016.parquet - 385.44 MB
train-00014-of-00016.parquet - 378.19 MB
train-00015-of-00016.parquet - 394.38 MB
Total dataset size: 6.22 GB


In [3]:
from datasets import load_dataset

data_files = {
    "train": "../Data/train-*.parquet",
    "test": "../Data/test-*.parquet"
}

ds = load_dataset(
    "parquet",
    data_files=data_files
)

print(ds)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 108309
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})


In [4]:
from collections import Counter

train_labels = Counter(ds["train"]["label"])
test_labels = Counter(ds["test"]["label"])

class_names = ds["train"].features["label"].names

print("TRAINING CLASS DISTRIBUTION")

for label, name in enumerate(class_names):
    print(f"{name}: {train_labels[label]}")

print("\nTESTING CLASS DISTRIBUTION")

for label, name in enumerate(class_names):
    print(f"{name}: {test_labels[label]}")

TRAINING CLASS DISTRIBUTION
CNV: 37205
DME: 11348
DRUSEN: 8616
NORMAL: 51140

TESTING CLASS DISTRIBUTION
CNV: 250
DME: 250
DRUSEN: 250
NORMAL: 250


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split

all_labels = np.array(ds["train"]["label"])

all_indices = np.arange(len(all_labels))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42,
    stratify=all_labels
)

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Testing samples:", len(ds["test"]))

Training samples: 97478
Validation samples: 10831
Testing samples: 1000


In [8]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.2022, 0.2022, 0.2022],
        std=[0.2133, 0.2133, 0.2133]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.2022, 0.2022, 0.2022],
        std=[0.2133, 0.2133, 0.2133]
    )
])

print("Training and evaluation transformations created!")

Training and evaluation transformations created!


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class OCTDataset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = int(self.indices[index])

        sample = self.dataset[actual_index]

        image = sample["image"].convert("RGB")
        label = sample["label"]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [10]:
train_dataset = OCTDataset(
    ds["train"], train_indices, train_transform
)

val_dataset = OCTDataset(
    ds["train"], val_indices, eval_transform
)

test_indices = np.arange(len(ds["test"]))

test_dataset = OCTDataset(
    ds["test"], test_indices, eval_transform
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 97478
Validation dataset: 10831
Testing dataset: 1000


In [11]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 12185
Validation batches: 1354
Testing batches: 125


In [12]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Labels:", labels)

Image batch shape: torch.Size([8, 3, 224, 224])
Label batch shape: torch.Size([8])
Image dtype: torch.float32
Labels: tensor([3, 0, 0, 0, 0, 2, 3, 0])


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("Images device:", images.device)
print("Labels device:", labels.device)
print("GPU memory allocated:", round(
    torch.cuda.memory_allocated() / 1024**2, 2
), "MB")

Device: cuda
GPU: NVIDIA GeForce GTX 1650
Images device: cuda:0
Labels device: cuda:0
GPU memory allocated: 4.59 MB


In [14]:
import torch.nn as nn

class CustomCNN1(nn.Module):
    def __init__(self):
        super(CustomCNN1, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 4)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

In [15]:
model1 = CustomCNN1().to(device)

sample_images, sample_labels = next(iter(train_loader))

sample_images = sample_images.to(device)

model1.eval()

with torch.no_grad():
    output = model1(sample_images)

print("Input Shape:", sample_images.shape)
print("Output Shape:", output.shape)

print(
    "Model Parameters:",
    sum(p.numel() for p in model1.parameters())
)

Input Shape: torch.Size([8, 3, 224, 224])
Output Shape: torch.Size([8, 4])
Model Parameters: 110724


In [16]:
from collections import Counter

train_class_counts = Counter(all_labels[train_indices])

class_counts = np.array([
    train_class_counts[i] for i in range(4)
])

class_weights = len(train_indices) / (4 * class_counts)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class Counts:", class_counts)
print("Class Weights:", class_weights)

Class Counts: [33485 10213  7754 46026]
Class Weights: tensor([0.7278, 2.3861, 3.1428, 0.5295], device='cuda:0')


In [17]:
import torch.optim as optim

criterion1 = nn.CrossEntropyLoss(weight=class_weights)

optimizer1 = optim.Adam(
    model1.parameters(),
    lr=0.001
)

print("Loss function:", criterion1)
print("Optimizer:", optimizer1)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [18]:
scaler = torch.amp.GradScaler("cuda")

print("Mixed precision scaler:", scaler)
print("CUDA device:", device)

Mixed precision scaler: <torch.amp.grad_scaler.GradScaler object at 0x0000024424C734D0>
CUDA device: cuda


In [19]:
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

print("Model directory:", model_dir.resolve())

Model directory: C:\Users\Umesh eshwar reddy\Desktop\p4\models


In [20]:
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()

    loss_sum = 0.0
    weight_sum = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(loader)

    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            individual_losses = nn.functional.cross_entropy(
                outputs.float(),
                labels,
                weight=class_weights,
                reduction="none"
            )

            loss_sum += individual_losses.sum().item()
            weight_sum += class_weights[labels].sum().item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{correct / total:.4f}"
        )

    return loss_sum / weight_sum, correct / total

In [21]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    loss_sum = 0.0
    weight_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            individual_losses = nn.functional.cross_entropy(
                outputs.float(),
                labels,
                weight=class_weights,
                reduction="none"
            )

            loss_sum += individual_losses.sum().item()
            weight_sum += class_weights[labels].sum().item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return loss_sum / weight_sum, correct / total

In [22]:
import time

model1.train()

benchmark_batches = 100
benchmark_start = time.time()

for batch_idx, (images, labels) in enumerate(train_loader):
    if batch_idx >= benchmark_batches:
        break

    images = images.to(device)
    labels = labels.to(device)

    optimizer1.zero_grad(set_to_none=True)

    with torch.amp.autocast("cuda", dtype=torch.float16):
        outputs = model1(images)
        loss = criterion1(outputs, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer1)
    scaler.update()

    if (batch_idx + 1) % 20 == 0:
        print(f"Completed batches: {batch_idx + 1}/{benchmark_batches}")

benchmark_time = time.time() - benchmark_start

print("Benchmark time:", round(benchmark_time, 2), "seconds")
print("Average time per batch:", round(benchmark_time / benchmark_batches, 3), "seconds")
print("GPU memory allocated:", round(torch.cuda.memory_allocated() / 1024**2, 2), "MB")
print("GPU memory reserved:", round(torch.cuda.memory_reserved() / 1024**2, 2), "MB")

Completed batches: 20/100
Completed batches: 40/100
Completed batches: 60/100
Completed batches: 80/100
Completed batches: 100/100
Benchmark time: 11.09 seconds
Average time per batch: 0.111 seconds
GPU memory allocated: 22.55 MB
GPU memory reserved: 222.0 MB


In [23]:
model1 = CustomCNN1().to(device)

criterion1 = nn.CrossEntropyLoss(weight=class_weights)

optimizer1 = optim.Adam(
    model1.parameters(),
    lr=0.001
)

scaler1 = torch.amp.GradScaler("cuda")

best_val_acc1 = 0.0

history_cnn1 = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

print("Fresh CNN1 initialized!")

Fresh CNN1 initialized!


In [24]:
epochs = 5

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")

    train_loss, train_acc = train_one_epoch(
        model1,
        train_loader,
        criterion1,
        optimizer1,
        device,
        scaler1
    )

    val_loss, val_acc = validate_one_epoch(
        model1,
        val_loader,
        criterion1,
        device
    )

    history_cnn1["train_loss"].append(train_loss)
    history_cnn1["train_acc"].append(train_acc)
    history_cnn1["val_loss"].append(val_loss)
    history_cnn1["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc1:
        best_val_acc1 = val_acc

        torch.save(
            model1.state_dict(),
            model_dir / "custom_cnn1_best.pth"
        )

        print("Best model checkpoint saved!")

print("\nCNN1 Training Completed!")
print("Best Validation Accuracy:", best_val_acc1)


Epoch 1/5


100%|██████████| 1354/1354 [01:43<00:00, 13.10it/s]


Train Loss: 1.1028 | Train Accuracy: 0.6938
Val Loss: 0.9334 | Val Accuracy: 0.7655
Best model checkpoint saved!

Epoch 2/5


100%|██████████| 1354/1354 [02:17<00:00,  9.87it/s]


Train Loss: 0.8862 | Train Accuracy: 0.7441
Val Loss: 0.7494 | Val Accuracy: 0.8015
Best model checkpoint saved!

Epoch 3/5


100%|██████████| 1354/1354 [01:49<00:00, 12.34it/s]


Train Loss: 0.6634 | Train Accuracy: 0.8024
Val Loss: 0.4862 | Val Accuracy: 0.8881
Best model checkpoint saved!

Epoch 4/5


100%|██████████| 1354/1354 [01:53<00:00, 11.89it/s]


Train Loss: 0.5466 | Train Accuracy: 0.8416
Val Loss: 0.4668 | Val Accuracy: 0.8777

Epoch 5/5


100%|██████████| 1354/1354 [01:32<00:00, 14.65it/s]


Train Loss: 0.4678 | Train Accuracy: 0.8664
Val Loss: 0.3797 | Val Accuracy: 0.9073
Best model checkpoint saved!

CNN1 Training Completed!
Best Validation Accuracy: 0.9073031114393869


In [25]:
checkpoint_path = model_dir / "custom_cnn1_best.pth"

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint size:", round(
    checkpoint_path.stat().st_size / 1024**2, 2
), "MB")
print("Best validation accuracy:", best_val_acc1)

Checkpoint exists: True
Checkpoint size: 0.43 MB
Best validation accuracy: 0.9073031114393869


In [26]:
import json

with open("../outputs/cnn1_history.json", "w") as f:
    json.dump(history_cnn1, f, indent=4)

print("CNN1 training history saved!")

CNN1 training history saved!
